In [ ]:
# ==========================================
# STEP 1: Install Required Libraries
# ==========================================
!pip install finnhub-python vaderSentiment openpyxl --quiet

import os
import datetime
import pandas as pd
import finnhub
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from google.colab import drive

# ==========================================
# STEP 2: Configuration & Credentials
# ==========================================
# TODO: Replace with your actual Finnhub API key
FINNHUB_API_KEY = "daubhk9r01qkvn4p8t70daubhk9r01qkvn4p8t7g"

# Initialize Finnhub client
client = finnhub.Client(api_key=FINNHUB_API_KEY)

# ==========================================
# STEP 3: Mount Google Drive & Define Path
# ==========================================
drive.mount('/content/drive')

# Folder path in your Google Drive
folder_path = '/content/drive/My Drive/Financial_News_Pipeline'
os.makedirs(folder_path, exist_ok=True)
excel_file_path = os.path.join(folder_path, 'Daily_news_master.xlsx')

# ==========================================
# STEP 4: Ingest Data from Finnhub API
# ==========================================
print("Fetching financial news from Finnhub...")
try:
    # Fetching general market news
    news_list = client.general_news('general', min_id=0)

    if not news_list:
        print("No news returned from API.")
    else:
        df_raw = pd.DataFrame(news_list)
        print(f"Successfully fetched {len(df_raw)} raw articles.")

except Exception as e:
    print(f"Error fetching data from Finnhub API: {e}")
    df_raw = pd.DataFrame()

# ==========================================
# STEP 5: Transform & Clean Data
# ==========================================
if not df_raw.empty:
    # Select and rename relevant columns
    # Finnhub general news fields usually include: category, datetime, headline, id, image, related, source, summary, url
    columns_to_keep = ['id', 'datetime', 'headline', 'summary', 'source', 'category', 'url']
    existing_cols = [col for col in columns_to_keep if col in df_raw.columns]
    df = df_raw[existing_cols].copy()

    # Convert UNIX timestamp to readable date-time format
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
    df['date'] = df['datetime'].dt.date

    # Clean text (handle missing summaries or headlines)
    df['headline'] = df['headline'].fillna('').astype(str).str.strip()
    df['summary'] = df['summary'].fillna('').astype(str).str.strip()

    # Combine headline and summary for richer sentiment analysis text
    df['text_for_sentiment'] = df['headline'] + ". " + df['summary']

    # ==========================================
    # STEP 6: Sentiment Analysis (NLP)
    # ==========================================
    analyzer = SentimentIntensityAnalyzer()

    # Calculate compound sentiment score (-1 to +1)
    df['sentiment_score'] = df['text_for_sentiment'].apply(
        lambda x: analyzer.polarity_scores(x)['compound']
    )

    # Categorize sentiment label
    df['sentiment_label'] = df['sentiment_score'].apply(
        lambda score: 'Positive' if score > 0.05 else ('Negative' if score < -0.05 else 'Neutral')
    )

    # Drop temporary text column
    df = df.drop(columns=['text_for_sentiment'])

    # ==========================================
    # STEP 7: Save and Append to Excel Storage
    # ==========================================
    # ==========================================
# STEP 7: Manage Daily Active File & Archive
# ==========================================
    if os.path.exists(excel_file_path):
            # Load existing master file, append new data, and drop duplicates based on article 'id'
            existing_df = pd.read_excel(excel_file_path)

            # Ensure correct date types if needed
            combined_df = pd.concat([existing_df, df], ignore_index=True)
            combined_df.drop_duplicates(subset=['id'], keep='last', inplace=True)

            combined_df.to_excel(excel_file_path, index=False)
            print(f"Appended new records. Total records in master file: {len(combined_df)}")
    else:
        # Create a new master excel file if it doesn't exist yet
        df.to_excel(excel_file_path, index=False)
        print(f"Created new master Excel file with {len(df)} records.")

    print(f"File saved successfully to: {excel_file_path}")
else:
    print("Pipeline skipped transformation/storage due to empty ingestion data.")

Mounted at /content/drive
Fetching financial news from Finnhub...
Successfully fetched 100 raw articles.
Appended new records. Total records in master file: 391
File saved successfully to: /content/drive/My Drive/Financial_News_Pipeline/Daily_news_master.xlsx


In [ ]:
df = pd.read_excel('/content/drive/My Drive/Financial_News_Pipeline/Daily_news_master.xlsx')
df.tail(5)


,id,datetime,headline,summary,source,category,url,date,sentiment_score,sentiment_label
386,8709725,2026-11-02 15:24:57,Gold rises over 2% from one-year low on softer...,Gold rises over 2% from one-year low on softer...,Reuters,business,https://news.google.com/rss/articles/CBMimwFBV...,2026-11-02,-0.4939,Negative
387,8711195,2026-11-02 14:03:44,"Oracle files to sell $20 billion in new stock,...",Oracle filed plans on Monday to issue stock ne...,CNBC,top news,https://www.cnbc.com/2026/11/02/oracle-files-t...,2026-11-02,0.3626,Positive
388,8711197,2026-11-02 14:00:28,SNAPSHOT US stock index futures jump as oil fa...,SNAPSHOT US stock index futures jump as oil fa...,Reuters,business,https://news.google.com/rss/articles/CBMixgFBV...,2026-11-02,0.0000,Neutral
389,8711519,2026-11-02 13:56:54,Jim Cramer's top 10 things to watch in the sto...,Marvell makes its blockbuster deal for Lumentu...,CNBC,top news,https://www.cnbc.com/investingclub/2026/11/02/...,2026-11-02,0.6908,Positive
390,8711194,2026-11-02 13:56:01,We're buying the dip in a stock after an expec...,We waited for the dust to settle after its ear...,CNBC,top news,https://www.cnbc.com/investingclub/2026/11/02/...,2026-11-02,0.0000,Neutral
